# Расчет метрик

Рассчитываем конверсию визитов в регистрации на данных API предыдущего шага. Боты исключаются по слову `bot` в `User-Agent`, а для каждого `visit_id` учитывается только последний визит.

In [ ]:
%run ./data_preparation.ipynb

In [ ]:
conversion_visits = api_visits.copy()
conversion_registrations = api_registrations.copy()

conversion_visits["datetime"] = pd.to_datetime(
    conversion_visits["datetime"], format="ISO8601", errors="raise"
)
conversion_registrations["datetime"] = pd.to_datetime(
    conversion_registrations["datetime"], format="ISO8601", errors="raise"
)

conversion_visits = conversion_visits.loc[
    ~conversion_visits["user_agent"].str.contains("bot", case=False, na=False)
].copy()

conversion_visits = (
    conversion_visits
    .sort_values("datetime")
    .drop_duplicates(subset="visit_id", keep="last")
)

conversion_visits["date_group"] = conversion_visits["datetime"].dt.normalize()
conversion_registrations["date_group"] = conversion_registrations["datetime"].dt.normalize()

In [ ]:
visits_grouped = (
    conversion_visits
    .groupby(["date_group", "platform"], as_index=False)
    .agg(visits=("visit_id", "size"))
)

registrations_grouped = (
    conversion_registrations
    .groupby(["date_group", "platform"], as_index=False)
    .agg(registrations=("user_id", "size"))
)

In [ ]:
conversion = visits_grouped.merge(
    registrations_grouped,
    on=["date_group", "platform"],
    how="outer",
)

conversion[["visits", "registrations"]] = (
    conversion[["visits", "registrations"]]
    .fillna(0)
    .astype(int)
)

conversion["conversion"] = (
    conversion["registrations"] / conversion["visits"] * 100
)

conversion = (
    conversion[["date_group", "platform", "visits", "registrations", "conversion"]]
    .sort_values(["date_group", "platform"])
    .reset_index(drop=True)
)

conversion.to_json("./conversion.json")
conversion